# FDSI benchmark: optimisation -- pareto (wh_loss/sv_loss, median aggregation)

Runs an Optuna hyperparameter search (`optimize_adapt_decomp_pooled_memory_pareto`) over
`wh_learning_rate`/`sv_learning_rate` (and the other entries in `DEFAULT_PARAM_SPACE`), pooling the
3 triangular conditions (`POOL_CONDITIONS`, ramp durations from 5 s to 40 s) so one search covers
all of them at once. Each trial's `AdaptDecomp` run is scored on **two** objectives together --
whitening loss and separation-vector loss, each the median across the pooled conditions -- so
Optuna returns a Pareto front of trials where neither objective can be improved without worsening
the other, rather than a single winner; `min(front, key=...sv_loss...)` then picks one front
member as this branch's winning config, separately for each `lr_mode`.

**Load-only by default** (`RUN_OPTIMISATION=False`) -- reads the cached `study.pkl` under
`FDSI_ROOT/adaptation/optimisation/<lr_mode>/tpe_pareto/`, does not re-run the search.

## Config

In [1]:
RUN_OPTIMISATION = False   # run the pooled tpe_pareto study, or reuse a cached study.pkl

import yaml
from pathlib import Path

with open('../../configs/data_configs/fdsi_benchmark_grid.yaml') as f:
    grid = yaml.safe_load(f)

FDSI_ROOT = Path(grid['fdsi_root'])
DATA_DIR, CAL_DIR, ADAPT_DIR = FDSI_ROOT / 'data', FDSI_ROOT / 'calibration', FDSI_ROOT / 'adaptation'
OPT_DIR = ADAPT_DIR / 'optimisation'

FS, EXT_FACT = grid['fs'], grid['ext_fact']
TOL_SPIKE_MS, ROA_CAL_TH = grid['tol_spike_ms'], grid['roa_cal_th']
CAL_END, ISO_DUR = int(grid['cal_duration_s'] * FS), int(grid['iso_duration_s'] * FS)

SUBJECTS, CONDITIONS, SNR_LEVELS = grid['subjects'], grid['conditions'], grid['snr_levels']
TRIANGULAR = [c for c in CONDITIONS if 'triangular' in c]

POOL_CONDITIONS = grid['pool_conditions']
HOLDOUT_CONDITIONS = [c for c in CONDITIONS if c not in POOL_CONDITIONS]
OPTIM_SUB, OPTIM_SNR, N_TRIALS = grid['optim_sub'], grid['optim_snr'], grid['n_trials']

BASE_CONFIG_DIR = Path('../../configs/adapt_configs')
BASE_CONFIG_PATH = BASE_CONFIG_DIR / 'default_muniverse.yaml'   # wh_sv_coupling=True; lr_mode overridden per branch below
LR_TOKEN_TO_MODE = {'lr_fixed': 'fixed', 'lr_relerror': 'rel_error'}

print(f'Apply-to-all grid: {len(SUBJECTS)} subjects x {len(CONDITIONS)} conditions x {len(SNR_LEVELS)} SNR levels '
      f'= {len(SUBJECTS) * len(CONDITIONS) * len(SNR_LEVELS)} recordings')

OBJECTIVES = ('wh_loss', 'sv_loss')   # DEFAULT_OBJECTIVES, spelled out here since the import lives in the next cell
SAMPLER_NAME = 'tpe_pareto'

Apply-to-all grid: 5 subjects x 5 conditions x 4 SNR levels = 100 recordings


## Data pool

The 3 pooled conditions' `PooledDatasetMemory`s, loaded via `load_data()` from `configs/data_configs/fdsi_pool_memory_example.yaml` -- see [optimisation.md](../../docs/optimisation.md#loading-a-pool-from-a-data_config-yaml).

In [2]:
import pickle
import shutil
import sys
from typing import Dict
sys.path.insert(0, '../..')
sys.path.insert(0, str(Path.cwd()))

import optuna
from adapt_decomp import AdaptationResult, load_data
from adapt_decomp.adaptation import AdaptConfig
from adapt_decomp.adaptation.optimize import (
    DEFAULT_PARAM_SPACE, DEFAULT_OBJECTIVES, optimize_adapt_decomp_pooled_memory_pareto,
)
from adapt_decomp.utils.plots import plot_optimisation_landscape_grid, plot_pareto_scatter_grid
import fdsi_common as fc

DATA_CONFIG_PATH = Path('../../configs/data_configs/fdsi_pool_memory_example.yaml')
with DATA_CONFIG_PATH.open() as f:
    data_config = yaml.safe_load(f)
pool = load_data(data_config)
assert set(pool) == set(POOL_CONDITIONS), pool.keys()
{name: tuple(cond.emg.shape) for name, cond in pool.items()}

{'triangular-ramp40s': (184320, 100),
 'triangular-ramp10s': (184320, 100),
 'triangular-ramp5s': (184320, 100)}

## Optimisation

In [3]:
pareto_fronts: Dict[str, list] = {}
studies: Dict[str, optuna.Study] = {}
best_configs: Dict[str, AdaptConfig] = {}
best_outputs: Dict[str, dict] = {}
selected_trials: Dict[str, optuna.trial.FrozenTrial] = {}

for lr_token, lr_mode in LR_TOKEN_TO_MODE.items():
    base_config = AdaptConfig.from_yaml(BASE_CONFIG_PATH)
    base_config.lr_mode = lr_mode   # default_muniverse.yaml is lr_mode='fixed' -- override for lr_relerror
    best_dir = OPT_DIR / lr_token / SAMPLER_NAME
    study_path = best_dir / 'study.pkl'

    n_complete = 0
    if study_path.exists():
        with open(study_path, 'rb') as f:
            cached_study = pickle.load(f)
        n_complete = sum(t.state.name == 'COMPLETE' for t in cached_study.trials)

    if n_complete >= N_TRIALS:
        print(f'{lr_token}/{SAMPLER_NAME}: loading cached study ({n_complete} complete trials).')
        study = cached_study
        front = study.best_trials
        chosen = min(front, key=lambda t: t.user_attrs['sv_loss'])
        best_config = AdaptConfig.from_yaml(BASE_CONFIG_PATH)
        best_config.lr_mode = lr_mode
        for k, v_ in chosen.params.items():
            setattr(best_config, k, v_)
        outputs = {c: AdaptationResult.load(best_dir / f'trial_{chosen.number}' / f'{c}.pkl')
                   for c in POOL_CONDITIONS if (best_dir / f'trial_{chosen.number}' / f'{c}.pkl').exists()}
    elif RUN_OPTIMISATION:
        shutil.rmtree(best_dir, ignore_errors=True)
        print(f'Running {lr_token}/{SAMPLER_NAME} ({N_TRIALS} trials, objectives={OBJECTIVES}) ...')
        outputs, best_config, front, study = optimize_adapt_decomp_pooled_memory_pareto(
            pool=pool, param_space=DEFAULT_PARAM_SPACE, objectives=OBJECTIVES,
            base_config=base_config, compute_roa=True, roa_kwargs={'tol_spike_ms': TOL_SPIKE_MS},
            n_trials=N_TRIALS, random_seed=42, best_result_path=str(best_dir),
        )
        chosen = min(front, key=lambda t: t.user_attrs['sv_loss'])
    else:
        raise FileNotFoundError(f'No cached {lr_token}/{SAMPLER_NAME} study and RUN_OPTIMISATION=False.')

    pareto_fronts[lr_token], studies[lr_token] = front, study
    best_configs[lr_token], best_outputs[lr_token] = best_config, outputs
    selected_trials[lr_token] = chosen
    print(f'  {lr_token}: front size={len(front)}  '
          f'wh_learning_rate={best_config.wh_learning_rate:.4g}  sv_learning_rate={best_config.sv_learning_rate:.4g}')

lr_fixed/tpe_pareto: loading cached study (50 complete trials).
  lr_fixed: front size=14  wh_learning_rate=0.04038  sv_learning_rate=0.002422
lr_relerror/tpe_pareto: loading cached study (50 complete trials).
  lr_relerror: front size=3  wh_learning_rate=0.001948  sv_learning_rate=0.01728


### Optimisation landscape -- RoA vs pooled wh_loss/sv_loss/total_loss

In [4]:
fig1 = plot_optimisation_landscape_grid(studies, {t: t for t in studies}, selected_trials)
fig1.show()

### Pareto front -- wh_loss vs sv_loss, coloured by RoA

In [5]:
fig2 = plot_pareto_scatter_grid(studies, pareto_fronts, selected_trials)
fig2.show()

## Promote the winning `lr_fixed` config

Saves this branch's winning **`lr_fixed`**-mode config as a first-class, reusable
`configs/adapt_configs/` entry -- `04a_pareto_application.ipynb` (and
`05_comparison_sv_loss_pareto_roa.ipynb`) read it back via `AdaptConfig.from_yaml(...)` instead of
reaching into `FDSI_ROOT/adaptation/optimisation/...`. Only `lr_fixed` is promoted this way;
`lr_relerror` still reads back from its own cached `config.yaml` under `FDSI_ROOT`, same as
before.

In [6]:
promoted_path = Path('../../configs/adapt_configs/optim_muniverse_fdsi_pareto.yaml')
best_configs['lr_fixed'].to_yaml(promoted_path)
print(f'Promoted lr_fixed config -> {promoted_path}')

Promoted lr_fixed config -> ..\..\configs\adapt_configs\optim_muniverse_fdsi_pareto.yaml
